# Collecte INSEE : population au carreau de 200 m (Filosofi 2021)

Télécharge les carreaux INSEE (Métropole, Martinique, Réunion), calcule le centre de chaque carreau en lon/lat et charge la table Delta `insee_carreaux_200m`.

**Prérequis** : un Lakehouse attaché comme Lakehouse par défaut du notebook.
Les fichiers bruts sont écrits dans `Files/raw/`, les tables Delta dans `Tables/`.

*Notebook généré depuis `src/ingestion/` par `fabric/build_notebooks.py` : modifier les scripts, pas ce notebook.*

In [ ]:
FORCE = False  # True pour retélécharger le fichier INSEE


In [ ]:
%pip install -q pyproj


## Outils communs

In [ ]:
"""Outils communs à la collecte : chemins, téléchargements HTTP, écriture des tables.

Les mêmes scripts tournent en local (sortie dans data/raw/) et dans un notebook
Microsoft Fabric (sortie dans le Lakehouse attaché : /lakehouse/default/Files/raw/).
"""
import json
import os
import time
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import requests

FABRIC_LAKEHOUSE = Path("/lakehouse/default/Files/raw")
LOCAL_RAW = Path(__file__).resolve().parents[2] / "data" / "raw" if "__file__" in globals() else Path("data/raw")

USER_AGENT = "hackathon-mobility-citiz/1.0 (collecte open data)"
PAN_API = "https://transport.data.gouv.fr/api/datasets"


def raw_dir() -> Path:
    """Dossier racine des données brutes : Lakehouse Fabric si présent, sinon data/raw/."""
    base = Path(os.environ["RAW_DIR"]) if "RAW_DIR" in os.environ else (
        FABRIC_LAKEHOUSE if FABRIC_LAKEHOUSE.parent.exists() else LOCAL_RAW
    )
    base.mkdir(parents=True, exist_ok=True)
    return base


def tables_dir() -> Path:
    """Tables Parquet prêtes à être chargées en tables Delta dans Fabric."""
    d = raw_dir() / "tables"
    d.mkdir(parents=True, exist_ok=True)
    return d


def now_utc() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def http_get(url: str, timeout: int = 60, retries: int = 3, **kwargs) -> requests.Response:
    headers = {"User-Agent": USER_AGENT, **kwargs.pop("headers", {})}
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=headers, timeout=timeout, **kwargs)
            r.raise_for_status()
            return r
        except requests.RequestException:
            if attempt == retries - 1:
                raise
            time.sleep(2 ** attempt)


def get_json(url: str, timeout: int = 30):
    return http_get(url, timeout=timeout).json()


def download(url: str, dest: Path, force: bool = False, version: str | None = None) -> Path:
    """Télécharge un fichier en streaming. Ne retélécharge pas s'il existe déjà, sauf si
    `version` (date de mise à jour, empreinte…) diffère de celle du fichier en cache."""
    dest.parent.mkdir(parents=True, exist_ok=True)
    fichier_version = dest.with_name(dest.name + ".version")
    en_cache = fichier_version.read_text() if fichier_version.exists() else None
    if dest.exists() and dest.stat().st_size > 0 and not force and (version is None or version == en_cache):
        print(f"  déjà présent : {dest.name}")
        return dest
    tmp = dest.with_suffix(dest.suffix + ".part")
    with http_get(url, timeout=300, stream=True) as r, open(tmp, "wb") as f:
        for chunk in r.iter_content(chunk_size=1 << 20):
            f.write(chunk)
    tmp.rename(dest)
    if version is not None:
        fichier_version.write_text(version)
    elif fichier_version.exists():
        fichier_version.unlink()
    print(f"  téléchargé : {dest.name} ({dest.stat().st_size / 1e6:.1f} Mo)")
    return dest


def save_json(obj, dest: Path) -> None:
    dest.parent.mkdir(parents=True, exist_ok=True)
    dest.write_text(json.dumps(obj, ensure_ascii=False))


def _manquant(v) -> bool:
    """Valeur manquante (None, NaN, pd.NA), sans échouer sur les listes ou dictionnaires des flux JSON."""
    return v is None or v is pd.NA or (isinstance(v, float) and v != v)


def _types_homogenes(df: pd.DataFrame) -> pd.DataFrame:
    """Les flux open data mélangent parfois nombres et textes dans une même colonne :
    on convertit ces colonnes en texte pour que Parquet / Delta les acceptent.
    Les colonnes booléennes avec des valeurs manquantes restent booléennes (type nullable)."""
    for col in df.columns[df.dtypes == object]:
        valeurs = df[col].dropna()
        types = {type(v) for v in valeurs}
        if not types:  # colonne entièrement vide : type "null" refusé par Delta, on la type en texte
            df[col] = df[col].astype("string")
        elif types & {bool, np.bool_} and types <= {bool, np.bool_, str} and valeurs.map(
                lambda v: str(v).lower() in ("true", "false")).all():
            # booléens avec des trous (ou déjà écrits en texte "True"/"False" dans l'historique)
            df[col] = df[col].map(lambda v: None if _manquant(v) else str(v).lower() == "true").astype("boolean")
        elif all(issubclass(t, (int, float, np.integer, np.floating)) and not issubclass(t, (bool, np.bool_))
                 for t in types):  # entiers et décimaux mélangés : nombre décimal
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
        elif len(types) > 1 or types - {str}:
            df[col] = df[col].map(lambda v: None if _manquant(v) else str(v))
    return df


# Tables écrites pendant l'exécution en cours : le notebook Fabric ne publie que celles-ci.
TABLES_ECRITES: list[str] = []


def write_table(df: pd.DataFrame, name: str, historique: bool = False) -> Path:
    """Écrit une table Parquet dans <raw>/tables/<name>.parquet.

    historique=True : la table ne contient que la photo de cette exécution ; elle est aussi
    archivée dans <raw>/historique/<name>/<date_collecte>.parquet, sans relire les photos
    précédentes (le notebook Fabric l'ajoute à la table Delta au lieu de la réécrire)."""
    path = tables_dir() / f"{name}.parquet"
    if df.empty and not historique and path.exists():  # collecte vide : on garde les dernières données valides
        print(f"  table {name} : aucune ligne collectée, version précédente conservée")
        if name not in TABLES_ECRITES:
            TABLES_ECRITES.append(name)
        return path
    df = _types_homogenes(df.copy())  # ne modifie pas le DataFrame de l'appelant
    df.to_parquet(path, index=False)
    if historique:
        archive = raw_dir() / "historique" / name / f"{datetime.now(timezone.utc):%Y%m%dT%H%M%S%fZ}.parquet"
        archive.parent.mkdir(parents=True, exist_ok=True)
        df.to_parquet(archive, index=False)
    if name not in TABLES_ECRITES:
        TABLES_ECRITES.append(name)
    print(f"  table {name} : {len(df):,} lignes -> {path}")
    return path


def reprendre_precedent(df: pd.DataFrame, name: str, ids_echec, cle: str = "ressource_id"):
    """Ajoute à df les lignes de la version précédente de la table pour les sources en échec :
    une panne passagère d'un flux ne fait pas disparaître ses dernières données valides.
    Ces lignes gardent leur date_collecte d'origine. Renvoie (table, identifiants repris)."""
    path = tables_dir() / f"{name}.parquet"
    ids = {str(i) for i in ids_echec}
    if not ids or not path.exists():
        return df, set()
    precedent = pd.read_parquet(path)
    if cle not in precedent:
        return df, set()
    precedent = precedent[precedent[cle].astype(str).isin(ids)]
    if precedent.empty:
        return df, set()
    print(f"  {name} : {len(precedent):,} lignes précédentes conservées pour {precedent[cle].nunique()} source(s) en échec")
    return pd.concat([df, precedent], ignore_index=True) if len(df.columns) else precedent, set(precedent[cle].astype(str))


def pan_catalog() -> list:
    """Catalogue complet du Point d'Accès National transport.data.gouv.fr."""
    return get_json(PAN_API, timeout=120)


## Collecte

In [ ]:
"""Collecte INSEE : population et revenus au carreau de 200 m (Filosofi 2021).

Source : https://www.insee.fr/fr/statistiques/8735162
Couverture : France métropolitaine, Martinique, La Réunion.

Sorties :
  <raw>/insee/Filosofi2021_carreaux_200m_csv.zip   (fichier brut INSEE)
  <raw>/tables/insee_carreaux_200m.parquet        (une ligne par carreau habité)
"""
import re
import zipfile

import pandas as pd


FILOSOFI_URL = "https://www.insee.fr/fr/statistiques/fichier/8735162/Filosofi2021_carreaux_200m_csv.zip"
TERRITOIRES = {"met": "Métropole", "mart": "Martinique", "reun": "La Réunion"}
ID_PATTERN = re.compile(r"CRS(\d+)RES200mN(\d+)E(\d+)")


def _ajouter_coordonnees(df: pd.DataFrame) -> pd.DataFrame:
    """Déduit la projection et le centre du carreau à partir de son identifiant INSEE.

    L'identifiant code le coin sud-ouest du carreau (ex. CRS3035RES200mN2029400E4259000) ;
    le centre est donc à +100 m sur chaque axe.
    """
    parts = df["idcar_200m"].str.extract(ID_PATTERN)
    df["epsg"] = parts[0].astype(int)
    df["x_centre"] = parts[2].astype(float) + 100
    df["y_centre"] = parts[1].astype(float) + 100
    try:
        from pyproj import Transformer
    except ImportError:
        print("  pyproj absent : lon/lat non calculées (x_centre / y_centre restent disponibles)")
        return df
    df["lon"], df["lat"] = float("nan"), float("nan")
    for epsg, idx in df.groupby("epsg").groups.items():
        t = Transformer.from_crs(f"EPSG:{epsg}", "EPSG:4326", always_xy=True)
        lon, lat = t.transform(df.loc[idx, "x_centre"].values, df.loc[idx, "y_centre"].values)
        df.loc[idx, "lon"], df.loc[idx, "lat"] = lon, lat
    return df


def collecter(force: bool = False) -> pd.DataFrame:
    print("INSEE Filosofi 2021 - carreaux 200 m")
    zip_path = download(FILOSOFI_URL, raw_dir() / "insee" / "Filosofi2021_carreaux_200m_csv.zip", force=force)

    frames = []
    with zipfile.ZipFile(zip_path) as z:
        for name in z.namelist():
            code = name.removeprefix("carreaux_200m_").removesuffix(".csv")
            with z.open(name) as f:
                df = pd.read_csv(f, dtype={"idcar_200m": str, "idcar_1km": str, "idcar_nat": str, "lcog_geo": str})
            df.insert(0, "territoire", TERRITOIRES.get(code, code))
            frames.append(df)
            print(f"  {name} : {len(df):,} carreaux")

    df = _ajouter_coordonnees(pd.concat(frames, ignore_index=True))
    df["source"] = "INSEE Filosofi 2021"
    df["date_collecte"] = now_utc()
    write_table(df, "insee_carreaux_200m")
    return df


In [ ]:
TABLES_ECRITES.clear()  # seules les tables de cette collecte seront publiées
collecter(force=FORCE)


## Chargement en tables Delta

In [ ]:
TABLES = ['insee_carreaux_200m']
HISTORIQUES = []

# Chargement en tables Delta du Lakehouse par défaut, à partir des Parquet écrits par CETTE exécution :
# - table produite : remplacée ; pour un historique, la photo est ajoutée à la suite (sans relire l'existant) ;
# - table non produite (ex. ZONE modifiée, AVEC_STOP_TIMES=False) : supprimée, pour ne jamais publier
#   d'anciennes données ; un historique sans nouvelle photo reste inchangé.
# L'écriture passe par l'adresse OneLake (abfss) : le dossier monté /lakehouse/default/Tables
# interrompt les envois de gros fichiers.
import shutil
from pathlib import Path
import pyarrow.parquet as pq
from deltalake import DeltaTable, Schema, write_deltalake

if not TABLES_ECRITES:
    raise RuntimeError("Aucune table produite dans cette session : exécuter d'abord la cellule de collecte.")

try:
    import notebookutils
    ctx = notebookutils.runtime.context
    TABLES_DELTA = (f"abfss://{ctx['defaultLakehouseWorkspaceId']}@onelake.dfs.fabric.microsoft.com/"
                    f"{ctx['defaultLakehouseId']}/Tables" + ("/dbo" if Path("/lakehouse/default/Tables/dbo").exists() else ""))
    OPTIONS = {"bearer_token": notebookutils.credentials.getToken("storage"), "use_fabric_endpoint": "true"}

    def supprimer(chemin):
        if notebookutils.fs.exists(chemin):
            notebookutils.fs.rm(chemin, True)
except ImportError:  # hors Fabric : tables Delta locales, pour tester
    TABLES_DELTA, OPTIONS = str(raw_dir() / "delta"), None

    def supprimer(chemin):
        shutil.rmtree(chemin, ignore_errors=True)

for nom in TABLES + HISTORIQUES:
    cible, source = f"{TABLES_DELTA}/{nom}", raw_dir() / "tables" / f"{nom}.parquet"
    if nom not in TABLES_ECRITES:
        if nom in HISTORIQUES:
            print(f"{nom} : pas de nouvelle photo, historique inchangé")
        else:
            supprimer(cible)
            source.unlink(missing_ok=True)
            print(f"{nom} : non produite par cette exécution, table supprimée")
        continue
    donnees = pq.read_table(source)
    if nom not in HISTORIQUES and donnees.num_rows == 0:  # collecte vide : on ne remplace pas des données valides
        print(f"{nom} : aucune ligne collectée, table précédente conservée")
        continue
    if nom not in HISTORIQUES:
        write_deltalake(cible, donnees, mode="overwrite", schema_mode="overwrite", storage_options=OPTIONS)
        print(f"{nom} : table Delta remplacée ({donnees.num_rows:,} lignes)")
        continue
    # photo déjà publiée (cellule relancée après un échec partiel) : pas de doublon
    photo = donnees.column("date_collecte")[0].as_py() if "date_collecte" in donnees.column_names and donnees.num_rows else None
    existe = DeltaTable.is_deltatable(cible, storage_options=OPTIONS)
    if existe and photo and len(DeltaTable(cible, storage_options=OPTIONS).to_pandas(
            columns=["date_collecte"], filters=[("date_collecte", "=", photo)])):
        print(f"{nom} : photo du {photo} déjà présente, rien à ajouter")
        continue
    # une colonne dont le type change (ex. booléens écrits en texte par une version précédente)
    # serait convertie silencieusement : on réécrit alors l'historique une fois, avec les types imposés
    if existe:
        existant = {f.name: str(f.type) for f in DeltaTable(cible, storage_options=OPTIONS).schema().fields}
        nouveau = {f.name: str(f.type) for f in (Schema.from_arrow(donnees.schema) if hasattr(Schema, "from_arrow")
                                                 else Schema.from_pyarrow(donnees.schema)).fields}
        conflits = [c for c in nouveau if c in existant and existant[c] != nouveau[c]]
    else:
        conflits = []
    if conflits:
        print(f"{nom} : types modifiés ({', '.join(conflits)}), réécriture unique de l'historique")
        ancien = DeltaTable(cible, storage_options=OPTIONS).to_pandas()
        tout = pd.concat([ancien.astype({c: object for c in conflits}), donnees.to_pandas()], ignore_index=True)
        if "_typer" in globals():  # types déclarés par le module de collecte (TYPES_HISTORIQUES du GBFS)
            tout = _typer(tout)
        write_deltalake(cible, _types_homogenes(tout), mode="overwrite", schema_mode="overwrite", storage_options=OPTIONS)
    else:
        write_deltalake(cible, donnees, mode="append", schema_mode="merge", storage_options=OPTIONS)
    print(f"{nom} : {donnees.num_rows:,} lignes ajoutées à l'historique")
